# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [2]:
%pip install polars duckdb fastexcel openpyxl pyarrow

  Using cached polars-2.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached duckdb-2.0.0-cp312-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached polars_runtime_32-2.0.0-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached polars-2.0.0-py3-none-any.whl (876 kB)
Using cached polars_runtime_32-2.0.0-cp310-abi3-win_amd64.whl (54.2 MB)
Using cached duckdb-2.0.0-cp312-abi3-win_amd64.whl (15.4 MB)
   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   ------------ --------------------------- 1.0/3.3 MB 12.9 MB/s eta 0:00:01
   ------------------------------------- -- 3.1/3.3 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 3.3/3.3 MB 9.6 MB/s  0:00:00

   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   ---------------------------------------- 0/4 [polars-runtime-32]
   -----------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from pathlib import Path
import polars as pl
import duckdb

# Jalankan notebook dari root project ATAU folder notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'online_retail_II.xlsx'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'{DATA_PATH} tidak ditemukan. Simpan Excel di data/raw/.')

# Jika modul belum tersedia, jalankan di terminal / cell terpisah:
# %pip install polars fastexcel duckdb
sheet_1 = pl.read_excel(DATA_PATH, sheet_name='Year 2009-2010', engine='calamine')
sheet_2 = pl.read_excel(DATA_PATH, sheet_name='Year 2010-2011', engine='calamine')

# Menyamakan tipe antar-sheet (misalnya Invoice string vs angka) agar concat aman.
# Kolom asal sheet membantu audit data, tetapi bukan kunci transaksi tunggal.
def standardisasi_sheet(frame, nama):
    frame = frame.with_columns([
        pl.col('Invoice').cast(pl.String, strict=False),
        pl.col('StockCode').cast(pl.String, strict=False),
        pl.col('Description').cast(pl.String, strict=False),
        pl.col('Country').cast(pl.String, strict=False),
        pl.col('Customer ID').cast(pl.String, strict=False),
        pl.col('Quantity').cast(pl.Int64, strict=False),
        pl.col('Price').cast(pl.Float64, strict=False),
        pl.col('InvoiceDate').cast(pl.Datetime, strict=False),
    ])
    return frame.with_columns(pl.lit(nama).alias('SourceSheet'))

df = pl.concat([
    standardisasi_sheet(sheet_1, 'Year 2009-2010'),
    standardisasi_sheet(sheet_2, 'Year 2010-2011'),
], how='diagonal_relaxed')
df = df.with_columns((pl.col('Quantity') * pl.col('Price')).alias('TotalPrice'))
print('Ukuran awal:', df.shape)
print('Kolom:', df.columns)
df.head()

print(df.shape)
print(df.head())

Ukuran awal: (1067371, 10)
Kolom: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'SourceSheet', 'TotalPrice']
(1067371, 10)
shape: (5, 10)
┌─────────┬───────────┬────────────┬──────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ Invoice ┆ StockCode ┆ Descriptio ┆ Quantity ┆ … ┆ Customer   ┆ Country   ┆ SourceShe ┆ TotalPric │
│ ---     ┆ ---       ┆ n          ┆ ---      ┆   ┆ ID         ┆ ---       ┆ et        ┆ e         │
│ str     ┆ str       ┆ ---        ┆ i64      ┆   ┆ ---        ┆ str       ┆ ---       ┆ ---       │
│         ┆           ┆ str        ┆          ┆   ┆ str        ┆           ┆ str       ┆ f64       │
╞═════════╪═══════════╪════════════╪══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 489434  ┆ 85048     ┆ 15CM       ┆ 12       ┆ … ┆ 13085      ┆ United    ┆ Year      ┆ 83.4      │
│         ┆           ┆ CHRISTMAS  ┆          ┆   ┆            ┆ Kingdom   ┆ 2009-2010 ┆           

## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [16]:
connection = duckdb.connect()
connection.register("dataset", df)

# Jalankan profiling
result = connection.execute(
    "SUMMARIZE SELECT * FROM dataset"
)

# Ambil hasil sebagai data Python
columns = [col[0] for col in result.description]
rows = result.fetchall()

# Konversi ke Polars
profil = pl.DataFrame(
    rows,
    schema=columns,
    orient="row"
)

profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""Invoice""","""VARCHAR""","""489434""","""581587""",47103,null,null,null,null,null,1067371,1.83
"""StockCode""","""VARCHAR""","""10002""","""m""",5573,null,null,null,null,null,1067371,0.00
"""Description""","""VARCHAR""",""" DOORMAT UNION JACK GUNS AND …","""wrongly sold sets""",5671,null,null,null,null,null,1067371,0.41
"""Quantity""","""BIGINT""","""-80995""","""80995""",1079,"""9.9388984711033""","""172.7057940767538""","""1""","""3""","""10""",1067371,0.00
"""InvoiceDate""","""TIMESTAMP""","""2009-12-01 07:45:00""","""2011-12-09 12:50:00""",47893,"""2011-01-02 21:13:55.394029""",null,"""2010-07-09 22:19:44.503925""","""2010-12-07 14:02:59.018381""","""2011-07-23 03:08:08.382223""",1067371,0.00
"""Price""","""DOUBLE""","""-53594.36""","""38970.0""",2753,"""4.649387727416289""","""123.55305872146295""","""1.25""","""2.095939892419811""","""4.145963007248118""",1067371,0.00
"""Customer ID""","""VARCHAR""","""12346""","""18287""",5986,null,null,null,null,null,1067371,22.77
"""Country""","""VARCHAR""","""Australia""","""West Indies""",44,null,null,null,null,null,1067371,0.00
"""SourceSheet""","""VARCHAR""","""Year 2009-2010""","""Year 2010-2011""",2,null,null,null,null,null,1067371,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [17]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns((pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null'))
    .sort('jumlah_null', descending=True)
)
print('Total baris:', f'{jumlah_baris:,}')
null_summary

Total baris: 1,067,371


kolom,jumlah_null,persen_null
str,u32,f64
"""Customer ID""",243007,22.77
"""Invoice""",19500,1.83
"""Description""",4382,0.41
"""StockCode""",0,0.0
"""Quantity""",0,0.0
"""InvoiceDate""",0,0.0
"""Price""",0,0.0
"""Country""",0,0.0
"""SourceSheet""",0,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [18]:
# Ubah teks kosong menjadi null dan rapikan spasi tanpa mengubah identitas transaksi.
df_normal = df.with_columns([
    pl.when(pl.col(c).str.strip_chars() == '')
        .then(None).otherwise(pl.col(c).str.strip_chars()).alias(c)
    for c in ['Invoice', 'StockCode', 'Description', 'Country', 'Customer ID']
])

KOLOM_WAJIB = ['Invoice', 'StockCode', 'InvoiceDate', 'Quantity', 'Price']
invalid_wajib = pl.any_horizontal([pl.col(c).is_null() for c in KOLOM_WAJIB])
df_dikeluarkan = df_normal.filter(invalid_wajib)
df_missing_handled = (
    df_normal.filter(~invalid_wajib)
    .with_columns([
        pl.col('Description').fill_null('Tidak diketahui'),
        pl.col('Country').fill_null('Tidak diketahui'),
    ])
)
print('Null awal total:', df.null_count().sum_horizontal().sum())
print('Baris tidak lengkap pada kolom wajib:', len(df_dikeluarkan))
print('Baris setelah seleksi:', len(df_missing_handled))
print('Null tersisa per kolom (Customer ID boleh kosong):')
print(df_missing_handled.null_count())

Null awal total: 266889
Baris tidak lengkap pada kolom wajib: 19500
Baris setelah seleksi: 1047871
Null tersisa per kolom (Customer ID boleh kosong):
shape: (1, 10)
┌─────────┬───────────┬────────────┬──────────┬───┬────────────┬─────────┬────────────┬────────────┐
│ Invoice ┆ StockCode ┆ Descriptio ┆ Quantity ┆ … ┆ Customer   ┆ Country ┆ SourceShee ┆ TotalPrice │
│ ---     ┆ ---       ┆ n          ┆ ---      ┆   ┆ ID         ┆ ---     ┆ t          ┆ ---        │
│ u32     ┆ u32       ┆ ---        ┆ u32      ┆   ┆ ---        ┆ u32     ┆ ---        ┆ u32        │
│         ┆           ┆ u32        ┆          ┆   ┆ u32        ┆         ┆ u32        ┆            │
╞═════════╪═══════════╪════════════╪══════════╪═══╪════════════╪═════════╪════════════╪════════════╡
│ 0       ┆ 0         ┆ 0          ┆ 0        ┆ … ┆ 242251     ┆ 0       ┆ 0          ┆ 0          │
└─────────┴───────────┴────────────┴──────────┴───┴────────────┴─────────┴────────────┴────────────┘


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [19]:
KOLOM_NUMERIK = 'Price'
harga_positif = df_missing_handled.filter(pl.col('Price') > 0)['Price']
if len(harga_positif) == 0:
    raise ValueError('Tidak ada Price positif untuk menghitung IQR.')
q1 = harga_positif.quantile(0.25)
q3 = harga_positif.quantile(0.75)
iqr = q3 - q1
batas_bawah = max(0.0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

# Flag hanya harga positif; nol dipertahankan sebagai kategori transaksi tersendiri.
is_outlier = ((pl.col('Price') > 0) &
                ((pl.col('Price') < batas_bawah) | (pl.col('Price') > batas_atas)))
df_outlier_handled = df_missing_handled.with_columns([
    is_outlier.alias('IsPriceOutlier'),
    pl.when(pl.col('Price') > 0)
        .then(pl.col('Price').clip(lower_bound=batas_bawah, upper_bound=batas_atas))
        .otherwise(pl.col('Price')).alias('Price_IQR_Capped'),
])
print(f'Q1={q1:.4f}, Q3={q3:.4f}, IQR={iqr:.4f}')
print(f'Batas IQR harga positif: {batas_bawah:.4f} s.d. {batas_atas:.4f}')
print('Jumlah outlier harga:', df_outlier_handled['IsPriceOutlier'].sum())
print('Jumlah Quantity negatif (potensi retur):', df_outlier_handled.filter(pl.col('Quantity') < 0).height)
print('Jumlah Price nol:', df_outlier_handled.filter(pl.col('Price') == 0).height)
print('Jumlah Price negatif (perlu audit):', df_outlier_handled.filter(pl.col('Price') < 0).height)
df_outlier_handled.select(['Price', 'Price_IQR_Capped', 'IsPriceOutlier']).head()

Q1=1.2500, Q3=4.1300, IQR=2.8800
Batas IQR harga positif: 0.0000 s.d. 8.4500
Jumlah outlier harga: 81632
Jumlah Quantity negatif (potensi retur): 3457
Jumlah Price nol: 6202
Jumlah Price negatif (perlu audit): 0


Price,Price_IQR_Capped,IsPriceOutlier
f64,f64,bool
6.95,6.95,false
6.75,6.75,false
6.75,6.75,false
2.1,2.1,false
1.25,1.25,false


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [20]:
KOLOM_KUNCI = ['SourceSheet', 'Invoice', 'StockCode', 'Description', 'Quantity',
                'InvoiceDate', 'Price', 'Customer ID', 'Country']
total_baris = len(df_outlier_handled)
# maintain_order=True mempertahankan kemunculan pertama.
df_dedup = df_outlier_handled.unique(subset=KOLOM_KUNCI, keep='first', maintain_order=True)
jumlah_duplikat = total_baris - len(df_dedup)
print(f'Total sebelum deduplikasi: {total_baris:,}')
print(f'Duplikat identik pada kunci gabungan: {jumlah_duplikat:,}')
print(f'Total sesudah deduplikasi: {len(df_dedup):,}')

Total sebelum deduplikasi: 1,047,871
Duplikat identik pada kunci gabungan: 12,072
Total sesudah deduplikasi: 1,035,799


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [23]:
# Koneksi khusus validasi
conn_validasi = duckdb.connect()
conn_validasi.register('dataset_bersih', df_dedup)

# Jalankan query validasi
result = conn_validasi.execute("""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE
            WHEN Invoice IS NULL
              OR StockCode IS NULL
              OR InvoiceDate IS NULL
              OR Quantity IS NULL
              OR Price IS NULL
            THEN 1 ELSE 0
        END) AS pelanggaran_null,

        SUM(CASE
            WHEN ABS(TotalPrice - Quantity * Price) > 0.000001
            THEN 1 ELSE 0
        END) AS pelanggaran_total_price,

        SUM(CASE
            WHEN Price > 0
             AND (Price_IQR_Capped < ?
                  OR Price_IQR_Capped > ?)
            THEN 1 ELSE 0
        END) AS pelanggaran_batas_iqr

    FROM dataset_bersih
""", [batas_bawah, batas_atas])

# Ambil hasil sebagai Polars DataFrame
columns = [col[0] for col in result.description]
rows = result.fetchall()

validasi = pl.DataFrame(
    rows,
    schema=columns,
    orient="row"
)

print(validasi)

shape: (1, 4)
┌─────────────┬──────────────────┬─────────────────────────┬───────────────────────┐
│ total_baris ┆ pelanggaran_null ┆ pelanggaran_total_price ┆ pelanggaran_batas_iqr │
│ ---         ┆ ---              ┆ ---                     ┆ ---                   │
│ i64         ┆ i64              ┆ i64                     ┆ i64                   │
╞═════════════╪══════════════════╪═════════════════════════╪═══════════════════════╡
│ 1035799     ┆ 0                ┆ 0                       ┆ 0                     │
└─────────────┴──────────────────┴─────────────────────────┴───────────────────────┘


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [24]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)
print('Baris sebelum cleaning:', f'{len(df):,}')
print('Baris dikeluarkan (kolom wajib null):', f'{len(df_dikeluarkan):,}')
print('Baris duplikat identik:', f'{jumlah_duplikat:,}')
print('Baris sesudah cleaning:', f'{len(df_dedup):,}')
print('Customer ID kosong yang dipertahankan:', df_dedup['Customer ID'].null_count())
print('Hasil:', OUTPUT_PATH)
print('Ukuran file (MB):', round(OUTPUT_PATH.stat().st_size / (1024**2), 2))

Baris sebelum cleaning: 1,067,371
Baris dikeluarkan (kolom wajib null): 19,500
Baris duplikat identik: 12,072
Baris sesudah cleaning: 1,035,799
Customer ID kosong yang dipertahankan: 242119
Hasil: C:\Users\User\Documents\semester 7\Big Data\tugas1-abd-2026\data\processed\dataset_bersih.parquet
Ukuran file (MB): 8.98


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

Kelengkapan (completeness): Audit missing values dilakukan per kolom. Description dan Country null diberi label Tidak diketahui; Customer ID null dipertahankan agar tidak membuat identitas pelanggan fiktif. Baris tanpa atribut transaksi wajib dipisahkan.

Validitas dan outlier: IQR dihitung dari Price positif. Harga asli tetap utuh; Price_IQR_Capped disediakan untuk analisis winsorized dan IsPriceOutlier untuk audit. Quantity negatif (potensi retur) dan Price nol tidak otomatis dibuang.

Keunikan: Duplikat ditentukan menggunakan gabungan atribut transaksi dan asal sheet, bukan nomor Invoice saja. Perlu 
diingat bahwa dua baris identik mungkin merupakan item transaksi yang benar-benar berbeda tanpa line-item ID.

Konsistensi: TotalPrice dihitung ulang dari Quantity * Price; skema kedua sheet diselaraskan.
Validasi dan keluaran: Pemeriksaan null wajib, duplikat, kesesuaian total harga, batas harga winsorized, dan rekonsiliasi jumlah baris dilakukan sebelum menyimpan dataset_bersih.parquet.